# RAG Assignment: Baseline vs. Re-Ranking
### 1. Setup & Environment

In [1]:
# Install required libraries
%pip install -q "transformers<5.0" "sentence-transformers<4.0" faiss-cpu pypdf accelerate pandas numpy torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import libraries
import os
import re
import time
import urllib.request
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import faiss
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version : {torch.__version__}")
print(f"Execution device: {DEVICE.upper()}")

PyTorch version : 2.5.1
Execution device: CPU


In [3]:
# Download the 2025 FIA Formula 1 Sporting Regulations PDF (if not already present) and chunk the text
PDF_PATH = "fia_f1_sporting_regulations.pdf"
PDF_URL = "https://api.fia.com/system/files/documents/fia_2025_formula_1_sporting_regulations_-_issue_5_-_2025-04-30.pdf"

if not os.path.exists(PDF_PATH):
    print(f"Downloading {PDF_PATH}...")
    req = urllib.request.Request(PDF_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response, open(PDF_PATH, "wb") as out_file:
        out_file.write(response.read())
    print("Download complete!")
else:
    print(f"Found PDF: {PDF_PATH} ({os.path.getsize(PDF_PATH) / 1e6:.2f} MB)")

reader = PdfReader(PDF_PATH)
print(f"Total PDF Pages: {len(reader.pages)}")

def chunk_pdf_document(pdf_reader, window_words=115, step_words=85, min_chars=120):
    # Split each page into overlapping word chunks
    corpus_chunks = []
    for page_idx, page in enumerate(pdf_reader.pages):
        raw_text = page.extract_text() or ""
        clean_text = re.sub(r"\s+", " ", raw_text).strip()
        words = clean_text.split(" ")
        for i in range(0, max(1, len(words) - 20), step_words):
            chunk_str = " ".join(words[i : i + window_words]).strip()
            if len(chunk_str) > min_chars:
                corpus_chunks.append({
                    "chunk_id": len(corpus_chunks),
                    "page": page_idx + 1,
                    "text": chunk_str
                })
    return corpus_chunks

chunks = chunk_pdf_document(reader, window_words=115, step_words=85)
print(f"Total Chunks Created: {len(chunks)}")

Found PDF: fia_f1_sporting_regulations.pdf (1.29 MB)
Total PDF Pages: 119
Total Chunks Created: 779


In [4]:
# Load Stage 1 Bi-Encoder, Stage 2 Cross-Encoder, and Local Open-Weights LLM
BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading Bi-Encoder...")
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device=DEVICE)

print("Loading Cross-Encoder...")
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device=DEVICE)

print(f"Loading Local LLM ({LLM_MODEL_NAME})...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
llm_dtype = torch.float16 if DEVICE == "cuda" else torch.float32
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    torch_dtype=llm_dtype
).to(DEVICE)
llm_model.eval()
print("All models loaded!")

Loading Bi-Encoder...
Loading Cross-Encoder...
Loading Local LLM (Qwen/Qwen2.5-0.5B-Instruct)...
All models loaded!


### 2. Two-Stage Retrieval Implementation

In [5]:
# Encode chunks with Bi-Encoder and index into FAISS (Cosine Similarity via normalized Inner Product)
t0 = time.perf_counter()
chunk_texts = [c["text"] for c in chunks]
corpus_embeddings = bi_encoder.encode(
    chunk_texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

embedding_dim = corpus_embeddings.shape[1]
vector_index = faiss.IndexFlatIP(embedding_dim)
vector_index.add(corpus_embeddings)
index_time_sec = time.perf_counter() - t0

print(f"Indexed {vector_index.ntotal} chunks (dim={embedding_dim}) in {index_time_sec:.2f}s")

Indexed 779 chunks (dim=384) in 14.13s


In [6]:
# Define LLM generation helper and Pipeline 1 (Baseline) vs. Pipeline 2 (Re-Ranking) functions
def generate_rag_answer(query: str, top3_chunks: list, max_new_tokens: int = 180) -> str:
    # Identical prompt structure used for both Pipeline 1 and Pipeline 2
    context_blocks = []
    for rank_idx, ch in enumerate(top3_chunks, start=1):
        context_blocks.append(
            f"[Context {rank_idx} | Chunk #{ch['chunk_id']}, Page {ch['page']}]:\n{ch['text']}"
        )
    formatted_context = "\n\n".join(context_blocks)

    system_msg = (
        "You are a regulatory assistant answering questions strictly from the provided "
        "FIA Formula 1 Sporting Regulations excerpts. State specific rules, exceptions, and numbers "
        "found in the context. If a detail is missing from the context, state only what is shown."
    )
    user_msg = (
        f"Retrieved Excerpts:\n{formatted_context}\n\n"
        f"Question: {query}\n\n"
        f"Answer based ONLY on the excerpts above:"
    )

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def run_two_stage_comparison(query: str, top_k_initial: int = 10, top_k_final: int = 3) -> dict:
    # Pipeline 1: Retrieve Top 10 chunks via Bi-Encoder cosine similarity and take Top 3
    t_start_bi = time.perf_counter()
    q_emb = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    bi_scores, bi_indices = vector_index.search(q_emb, top_k_initial)
    baseline_latency_ms = (time.perf_counter() - t_start_bi) * 1000.0

    candidates = []
    for rank_idx, (c_idx, b_score) in enumerate(zip(bi_indices[0].tolist(), bi_scores[0].tolist()), start=1):
        candidates.append({
            "chunk_id": int(c_idx),
            "page": chunks[c_idx]["page"],
            "text": chunks[c_idx]["text"],
            "baseline_rank": rank_idx,
            "bi_score": float(b_score)
        })

    baseline_top3 = candidates[:top_k_final]

    # Pipeline 2: Pass all 10 (query, chunk) pairs through the Cross-Encoder and re-sort descending
    t_start_ce = time.perf_counter()
    query_chunk_pairs = [[query, c["text"]] for c in candidates]
    ce_scores = cross_encoder.predict(query_chunk_pairs).tolist()
    for c, ce_s in zip(candidates, ce_scores):
        c["ce_score"] = float(ce_s)

    reranked_candidates = sorted(candidates, key=lambda x: x["ce_score"], reverse=True)
    for new_rank, c in enumerate(reranked_candidates, start=1):
        c["reranked_rank"] = new_rank
        c["rank_shift"] = c["baseline_rank"] - new_rank

    rerank_stage_ms = (time.perf_counter() - t_start_ce) * 1000.0
    total_pipeline2_retrieval_ms = baseline_latency_ms + rerank_stage_ms

    reranked_top3 = reranked_candidates[:top_k_final]

    # Generate Answer 1 (Baseline Top 3) and Answer 2 (Re-Ranked Top 3)
    answer_1 = generate_rag_answer(query, baseline_top3)
    answer_2 = generate_rag_answer(query, reranked_top3)

    return {
        "query": query,
        "baseline_latency_ms": baseline_latency_ms,
        "rerank_only_latency_ms": rerank_stage_ms,
        "pipeline2_retrieval_ms": total_pipeline2_retrieval_ms,
        "baseline_top3": baseline_top3,
        "reranked_top3": reranked_top3,
        "reranked_all10": reranked_candidates,
        "answer_1": answer_1,
        "answer_2": answer_2
    }

### 3. Failure Mode Analysis (Stress Testing)

In [7]:
# Evaluate failure queries where Baseline Retrieval misses vital chunks in Top 3, and Cross-Encoder restores them
stress_test_queries = [
    {
        "id": "Q1",
        "category": "Exceptions & Corner Cases",
        "query": "If a driver receives a five-second or ten-second time penalty during the final three laps of the race, why does the pit stop serving rule not apply and what happens instead?"
    },
    {
        "id": "Q2",
        "category": "Negation & Constraints",
        "query": "When a car is stopped in the pit stop position serving a time penalty, what are team mechanics strictly not permitted to do until the penalty time has elapsed?"
    },
    {
        "id": "Q3",
        "category": "Keyword Trap / Distractor",
        "query": "Under Parc Ferme regulations, why is a competitor not allowed to modify aerodynamic or suspension settings after qualifying unless climatic conditions change?"
    },
    {
        "id": "Q4",
        "category": "Other Document-Specific Failure",
        "query": "How many championship points are awarded to the top finishers if a race is suspended and cannot be resumed after completing more than two laps but less than 25% of the scheduled distance?"
    }
]

all_results = []

for item in stress_test_queries:
    print("=" * 100)
    print(f"[{item['id']}] {item['category']}: {item['query']}")
    print("-" * 100)

    res = run_two_stage_comparison(item["query"])
    res["id"] = item["id"]
    res["category"] = item["category"]
    all_results.append(res)

    rows = []
    for c in res["reranked_all10"]:
        shift = c["rank_shift"]
        shift_str = f"+{shift}" if shift > 0 else str(shift)
        rows.append({
            "Chunk ID": f"#{c['chunk_id']} (p.{c['page']})",
            "Baseline Rank": f"#{c['baseline_rank']}",
            "Bi Score": round(c["bi_score"], 4),
            "CE Score": round(c["ce_score"], 4),
            "Re-Ranked Pos": f"#{c['reranked_rank']}",
            "Shift": shift_str,
            "Base Top-3": "YES" if c["baseline_rank"] <= 3 else "no",
            "CE Top-3": "YES" if c["reranked_rank"] <= 3 else "no",
            "Snippet": c["text"][:90] + "..."
        })

    df_ranks = pd.DataFrame(rows)
    print("\nRank Shift Table (Top 10 Candidates):")
    print(df_ranks.to_string(index=False))

    print(f"\n[Answer 1 - Baseline Top 3]:\n{res['answer_1']}")
    print(f"\n[Answer 2 - Re-Ranked Top 3]:\n{res['answer_2']}\n")

[Q1] Exceptions & Corner Cases: If a driver receives a five-second or ten-second time penalty during the final three laps of the race, why does the pit stop serving rule not apply and what happens instead?
----------------------------------------------------------------------------------------------------

Rank Shift Table (Top 10 Candidates):
   Chunk ID Baseline Rank  Bi Score  CE Score Re-Ranked Pos Shift Base Top-3 CE Top-3                                                                                       Snippet
#431 (p.63)            #1    0.7113    4.8851            #1     0        YES      YES the pit lane, stop in his pit stop position for at least five seconds and then re- join th...
#435 (p.63)            #9    0.6157    4.2786            #2    +7         no      YES penalties above are imposed during the last three (3) laps, or after the end of a sprint s...
#434 (p.63)           #10    0.5994    3.7658            #3    +7         no      YES the sprint session or the ra

### 4. Latency Profiling

In [8]:
# Tabulate runtime of Baseline Retrieval vs. Re-Ranking Pipeline
latency_rows = []
for r in all_results:
    latency_rows.append({
        "Query ID": r["id"],
        "Category": r["category"],
        "Baseline Retrieval (ms)": round(r["baseline_latency_ms"], 2),
        "Cross-Encoder Re-Rank (ms)": round(r["rerank_only_latency_ms"], 2),
        "Total Re-Ranking Pipeline (ms)": round(r["pipeline2_retrieval_ms"], 2)
    })

df_latency = pd.DataFrame(latency_rows)
avg_row = {
    "Query ID": "MEAN",
    "Category": "Average",
    "Baseline Retrieval (ms)": round(df_latency["Baseline Retrieval (ms)"].mean(), 2),
    "Cross-Encoder Re-Rank (ms)": round(df_latency["Cross-Encoder Re-Rank (ms)"].mean(), 2),
    "Total Re-Ranking Pipeline (ms)": round(df_latency["Total Re-Ranking Pipeline (ms)"].mean(), 2)
}
df_latency = pd.concat([df_latency, pd.DataFrame([avg_row])], ignore_index=True)
print(df_latency.to_string(index=False))

Query ID                        Category  Baseline Retrieval (ms)  Cross-Encoder Re-Rank (ms)  Total Re-Ranking Pipeline (ms)
      Q1       Exceptions & Corner Cases                    21.36                      583.79                          605.14
      Q2          Negation & Constraints                    50.51                      255.26                          305.78
      Q3       Keyword Trap / Distractor                    17.99                      353.20                          371.19
      Q4 Other Document-Specific Failure                    29.54                      335.30                          364.84
    MEAN                         Average                    29.85                      381.89                          411.74


In [9]:
# Compare Top-10 Re-Ranking latency vs. running Cross-Encoder across all N chunks in the corpus
sample_q = stress_test_queries[0]["query"]
t_full = time.perf_counter()
all_pairs = [[sample_q, c["text"]] for c in chunks]
_ = cross_encoder.predict(all_pairs, batch_size=64, show_progress_bar=False)
full_corpus_ce_ms = (time.perf_counter() - t_full) * 1000.0

mean_p2_ms = df_latency.loc[df_latency["Query ID"] == "MEAN", "Total Re-Ranking Pipeline (ms)"].values[0]
print(f"Two-Stage Re-Ranking (Top 10 chunks) : {mean_p2_ms:.2f} ms")
print(f"Full-Corpus Cross-Encoder ({len(chunks)} chunks): {full_corpus_ce_ms:.2f} ms ({full_corpus_ce_ms/1000:.2f} s)")

Two-Stage Re-Ranking (Top 10 chunks) : 411.74 ms
Full-Corpus Cross-Encoder (779 chunks): 22524.27 ms (22.52 s)
